# Step 1：基础函数模块

本 Notebook 提供 Black-76 定价、解析 Greeks、Forward-to-Spot 转换、曲面风险差分接口、合约代码与日期解析、OTM 判断和 Delta 执行价搜索。

它不读取项目数据，也不包含波动率拟合、策略、调仓或 PnL 逻辑。

## 当前模块参数值

参数默认值统一在 `00_config.ipynb` 设置；本 cell 只打印当前内核中的实际值。


In [ ]:
_module_parameter_names = ["TRADING_DAYS_PER_YEAR", "EXPIRY_DATE_OVERRIDES"]
print(f'02_basic_functions.ipynb 当前参数：')
for _parameter_name in _module_parameter_names:
    print(f'{_parameter_name} = {globals()[_parameter_name]!r}')


In [ ]:
import calendar
import math
import re
from datetime import date, datetime
from typing import Callable, Iterable, Mapping, Optional

import pandas as pd

SQRT_2PI = math.sqrt(2.0 * math.pi)

def norm_pdf(x: float) -> float:
    return math.exp(-0.5 * x * x) / SQRT_2PI

def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def _normalize_option_type(option_type: str) -> str:#统一项目中Call/Put输入格式
    value = str(option_type).strip().upper()
    aliases = {'C': 'CALL', 'CALL': 'CALL', 'P': 'PUT', 'PUT': 'PUT'}
    if value not in aliases:
        raise ValueError("option_type must be 'CALL'/'C' or 'PUT'/'P'")
    return aliases[value]

def _validate_black76_inputs(F: float, K: float, sigma: float, tau: float) -> None:
    if F <= 0 or K <= 0:
        raise ValueError('F and K must be positive')
    if sigma <= 0:
        raise ValueError('sigma must be positive')
    if tau <= 0:
        raise ValueError('tau must be positive')

def black76_d1_d2(F: float, K: float, sigma: float, tau: float) -> tuple[float, float]:
    _validate_black76_inputs(F, K, sigma, tau)
    vol_sqrt_tau = sigma * math.sqrt(tau)
    d1 = (math.log(F / K) + 0.5 * sigma * sigma * tau) / vol_sqrt_tau
    return d1, d1 - vol_sqrt_tau

## Black-76 定价与解析 Greeks

`Delta` 和 `Gamma` 默认先按 Forward 求导。`Rho`、`RepoRho` 和 `Theta` 采用最终 Spot 风险口径：固定 Spot 及其他风险因子，对完整函数 $V(F(S,r,q,\tau),\tau,r)$ 求偏导。Theta 定义为 $\partial V/\partial\tau$。

In [ ]:
def black76_price(F: float, K: float, sigma: float, tau: float, r: float, option_type: str) -> float:
    option_type = _normalize_option_type(option_type)
    d1, d2 = black76_d1_d2(F, K, sigma, tau)
    discount = math.exp(-r * tau)
    if option_type == 'CALL':
        return discount * (F * norm_cdf(d1) - K * norm_cdf(d2))
    return discount * (K * norm_cdf(-d2) - F * norm_cdf(-d1))

def black76_delta(F: float, K: float, sigma: float, tau: float, r: float, option_type: str) -> float:
    """Forward delta: partial V / partial F."""
    option_type = _normalize_option_type(option_type)
    d1, _ = black76_d1_d2(F, K, sigma, tau)
    discount = math.exp(-r * tau)
    return discount * norm_cdf(d1) if option_type == 'CALL' else -discount * norm_cdf(-d1)

def black76_gamma(F: float, K: float, sigma: float, tau: float, r: float) -> float:
    """Forward gamma: partial squared V / partial F squared."""
    d1, _ = black76_d1_d2(F, K, sigma, tau)
    return math.exp(-r * tau) * norm_pdf(d1) / (F * sigma * math.sqrt(tau))

def black76_vega(F: float, K: float, sigma: float, tau: float, r: float) -> float:
    """partial V / partial sigma; sigma is expressed as a decimal, not vol points."""
    d1, _ = black76_d1_d2(F, K, sigma, tau)
    return math.exp(-r * tau) * F * norm_pdf(d1) * math.sqrt(tau)

def black76_volga(F: float, K: float, sigma: float, tau: float, r: float) -> float:
    """partial squared V / partial sigma squared."""
    d1, d2 = black76_d1_d2(F, K, sigma, tau)
    return black76_vega(F, K, sigma, tau, r) * d1 * d2 / sigma

def black76_vanna(F: float, K: float, sigma: float, tau: float, r: float, S: float) -> float:
    """Spot vanna: partial squared V / (partial S partial sigma), holding r, q and tau fixed."""
    if S <= 0:
        raise ValueError('S must be positive')
    d1, d2 = black76_d1_d2(F, K, sigma, tau)
    return -(F / S) * math.exp(-r * tau) * norm_pdf(d1) * d2 / sigma

def forward_delta_to_spot(delta_forward: float, F: float, S: float) -> float:
    if S <= 0:
        raise ValueError('S must be positive')
    return delta_forward * F / S

def forward_gamma_to_spot(gamma_forward: float, F: float, S: float) -> float:
    if S <= 0:
        raise ValueError('S must be positive')
    return gamma_forward * (F / S) ** 2

def black76_repo_rho(F: float, K: float, sigma: float, tau: float, r: float, option_type: str) -> float:
    """Spot-basis RepoRho = partial V / partial q = -tau * F * Delta_F."""
    return -tau * F * black76_delta(F, K, sigma, tau, r, option_type)

def black76_rho(F: float, K: float, sigma: float, tau: float, r: float, option_type: str) -> float:
    """Spot-basis total rho, including partial F / partial r = tau * F."""
    option_type = _normalize_option_type(option_type)
    _, d2 = black76_d1_d2(F, K, sigma, tau)
    discount = math.exp(-r * tau)
    return tau * K * discount * norm_cdf(d2) if option_type == 'CALL' else -tau * K * discount * norm_cdf(-d2)

def black76_theta(F: float, K: float, sigma: float, tau: float, r: float, q: float, option_type: str) -> float:
    """partial V / partial tau at fixed S, r, q and sigma; calendar passage uses negative Delta tau."""
    option_type = _normalize_option_type(option_type)
    d1, d2 = black76_d1_d2(F, K, sigma, tau)
    discount = math.exp(-r * tau)
    common = discount * F * norm_pdf(d1) * sigma / (2.0 * math.sqrt(tau))
    if option_type == 'CALL':
        return common - q * discount * F * norm_cdf(d1) + r * discount * K * norm_cdf(d2)
    return common + q * discount * F * norm_cdf(-d1) - r * discount * K * norm_cdf(-d2)

## 曲面 Greeks 的 bump-and-revalue 接口

`reprice(shocks)` 由后续波动率模块提供，返回固定实际合约集合在给定因子扰动下的组合价值。ATM、Skew、Curvature 使用中心差分；曲面二阶和交叉风险使用二阶中心差分。

In [ ]:
def _central_first(reprice: Callable[[dict], float], factor: str, h: float) -> float:
    if h <= 0:
        raise ValueError('bump size must be positive')
    return (reprice({factor: h}) - reprice({factor: -h})) / (2.0 * h)

def _central_second(reprice: Callable[[dict], float], factor: str, h: float) -> float:
    if h <= 0:
        raise ValueError('bump size must be positive')
    base = reprice({})
    return (reprice({factor: h}) - 2.0 * base + reprice({factor: -h})) / (h * h)

def _central_cross(reprice: Callable[[dict], float], x: str, hx: float, y: str, hy: float) -> float:
    if hx <= 0 or hy <= 0:
        raise ValueError('bump sizes must be positive')
    return (
        reprice({x: hx, y: hy}) - reprice({x: hx, y: -hy})
        - reprice({x: -hx, y: hy}) + reprice({x: -hx, y: -hy})
    ) / (4.0 * hx * hy)

def calculate_vol_greeks(
    reprice: Callable[[dict], float],
    atm_bump: float = 1e-4,
    skew_bump: float = 1e-4,
    curvature_bump: float = 1e-4,
    spot_bump: Optional[float] = None,
) -> dict:
    """Calculate model-independent surface-factor risks from a repricing callback.

    Shocks are absolute factor changes. The callback must interpret keys
    'ATM', 'Skew', 'Curvature', and optionally 'Spot'.
    """
    result = {
        'ATM': _central_first(reprice, 'ATM', atm_bump),
        'Skew': _central_first(reprice, 'Skew', skew_bump),
        'Curvature': _central_first(reprice, 'Curvature', curvature_bump),
        'VolgaSkew2': _central_second(reprice, 'Skew', skew_bump),
        'VolgaSkewCurvature': _central_cross(reprice, 'Skew', skew_bump, 'Curvature', curvature_bump),
        'VolgaCurvature2': _central_second(reprice, 'Curvature', curvature_bump),
    }
    if spot_bump is not None:
        result['VannaSkew'] = _central_cross(reprice, 'Spot', spot_bump, 'Skew', skew_bump)
        result['VannaCurvature'] = _central_cross(reprice, 'Spot', spot_bump, 'Curvature', curvature_bump)
    return result

In [ ]:
# 统一返回组合优化需要的单期权Spot Gamma、Vega及曲面一阶风险。
def get_option_risk_vector(
    F: float, S: float, K: float, sigma: float, tau: float, r: float,
    option_type: str, surface_reprice: Callable[[dict], float],
    skew_bump: float = 1e-4, curvature_bump: float = 1e-4,
) -> dict:
    """解析计算Gamma/Vega，并用中心差分计算SkewGreek与CurvatureGreek。"""
    delta_forward = black76_delta(F, K, sigma, tau, r, option_type)
    delta_spot = forward_delta_to_spot(delta_forward, F, S)
    gamma_forward = black76_gamma(F, K, sigma, tau, r)
    gamma_spot = forward_gamma_to_spot(gamma_forward, F, S)
    vega = black76_vega(F, K, sigma, tau, r)
    skew_greek = _central_first(surface_reprice, 'Skew', skew_bump)
    curvature_greek = _central_first(surface_reprice, 'Curvature', curvature_bump)
    return {
        'DELTA': float(delta_spot),
        'GAMMA': float(gamma_spot),
        'VEGA': float(vega),
        'SKEW': float(skew_greek),
        'CURVATURE': float(curvature_greek),
    }


In [ ]:
# 统一计算单期权全部Spot口径基础Greeks和波动率曲面Greeks。
def calculate_option_greeks(
    F: float, S: float, K: float, sigma: float, tau: float, r: float, q: float,
    option_type: str, surface_reprice: Optional[Callable[[dict], float]] = None,
    skew_bump: float = 1e-4, curvature_bump: float = 1e-4,
    surface_method: str = 'BUMP_AND_REVALUE',
) -> dict:
    """返回基础Spot Greeks、ATMVol二阶风险及Skew和Curvature风险。"""
    option_type = _normalize_option_type(option_type)
    delta_f = black76_delta(F, K, sigma, tau, r, option_type)
    delta_s = forward_delta_to_spot(delta_f, F, S)
    gamma_s = forward_gamma_to_spot(black76_gamma(F, K, sigma, tau, r), F, S)
    vega = black76_vega(F, K, sigma, tau, r)
    k = math.log(K / F)
    method = str(surface_method).upper()
    if method == 'ANALYTIC':
        skew_greek = vega * k
        curvature_greek = 0.5 * vega * k * k
    elif method in {'BUMP', 'BUMP_AND_REVALUE', 'FINITE_DIFFERENCE'}:
        if surface_reprice is None:
            raise ValueError('Bump-and-revalue曲面Greeks需要surface_reprice回调')
        skew_greek = _central_first(surface_reprice, 'Skew', skew_bump)
        curvature_greek = _central_first(surface_reprice, 'Curvature', curvature_bump)
    else:
        raise ValueError('surface_method必须为ANALYTIC或BUMP_AND_REVALUE')
    return {
        'DELTA': float(delta_s),
        'GAMMA': float(gamma_s),
        'VEGA': float(vega),
        'ATM_VOL_VANNA': float(black76_vanna(F, K, sigma, tau, r, S)),
        'ATM_VOL_VOLGA': float(black76_volga(F, K, sigma, tau, r)),
        'THETA': float(black76_theta(F, K, sigma, tau, r, q, option_type)),
        'RHO': float(black76_rho(F, K, sigma, tau, r, option_type)),
        'REPO_RHO': float(black76_repo_rho(F, K, sigma, tau, r, option_type)),
        'SKEW_GREEK': float(skew_greek),
        'CURVATURE_GREEK': float(curvature_greek),
    }


## 合约、日期、OTM 与 Delta 搜索工具

In [ ]:
OPTION_CODE_PATTERN = re.compile(r'^MO(?P<expiry>\d{4}|\d{6})-(?P<type>[CP])-(?P<strike>\d+(?:\.\d+)?)\.CFE$')
FUTURE_CODE_PATTERN = re.compile(r'^IM(?P<expiry>\d{4}|\d{6})\.CFE$')

def _parse_expiry_token(expiry: str) -> tuple[int, int]:
    if len(expiry) == 4:
        year, month = 2000 + int(expiry[:2]), int(expiry[2:])
    elif len(expiry) == 6:
        year, month = int(expiry[:4]), int(expiry[4:])
    else:
        raise ValueError('expiry must be yymm or yyyymm')
    if not 1 <= month <= 12:
        raise ValueError(f'invalid expiry month: {month}')
    return year, month

def parse_option_code(code: str) -> dict:
    match = OPTION_CODE_PATTERN.fullmatch(str(code).strip().upper())
    if not match:
        raise ValueError(f'invalid MO option code: {code}')
    expiry = match.group('expiry')
    year, month = _parse_expiry_token(expiry)
    return {
        'expiry_code': expiry, 'expiry_year': year, 'expiry_month': month,
        'option_type': 'CALL' if match.group('type') == 'C' else 'PUT',
        'strike': float(match.group('strike')),
    }

def parse_future_code(code: str) -> dict:
    match = FUTURE_CODE_PATTERN.fullmatch(str(code).strip().upper())
    if not match:
        raise ValueError(f'invalid IM future code: {code}')
    expiry = match.group('expiry')
    year, month = _parse_expiry_token(expiry)
    return {'expiry_code': expiry, 'expiry_year': year, 'expiry_month': month}

def _to_date(value) -> date:
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    return datetime.strptime(str(value), '%Y-%m-%d').date()

def calculate_expiry_date(expiry_code: str, overrides: Optional[Mapping[str, str]] = None) -> date:
    """Return override date first; otherwise return the third Friday."""
    token = str(expiry_code)
    if overrides and token in overrides:
        return _to_date(overrides[token])
    year, month = _parse_expiry_token(token)
    month_calendar = calendar.monthcalendar(year, month)
    fridays = [week[calendar.FRIDAY] for week in month_calendar if week[calendar.FRIDAY] != 0]
    return date(year, month, fridays[2])

def calculate_tau(valuation_date, expiry_date, days_per_year: int = 365) -> float:
    if days_per_year <= 0:
        raise ValueError('days_per_year must be positive')
    days = (_to_date(expiry_date) - _to_date(valuation_date)).days
    return max(days, 0) / float(days_per_year)

def is_otm_option(F: float, K: float, option_type: str, include_atm: bool = False) -> bool:
    option_type = _normalize_option_type(option_type)
    if option_type == 'CALL':
        return K >= F if include_atm else K > F
    return K <= F if include_atm else K < F

def find_strike_by_delta(
    strikes: Iterable[float],
    target_delta: float,
    F: float,
    S: float,
    tau: float,
    r: float,
    option_type: str,
    volatility_fn: Callable[[float], float],
) -> dict:
    """Return the listed strike whose model spot delta is closest to target_delta.

    volatility_fn receives log-moneyness k=log(K/F). The later volatility
    module supplies the fitted SVI or quadratic curve through this callback.
    """
    option_type = _normalize_option_type(option_type)
    candidates = []
    for strike in strikes:
        strike = float(strike)
        k = math.log(strike / F)
        sigma = float(volatility_fn(k))
        delta_f = black76_delta(F, strike, sigma, tau, r, option_type)
        delta_s = forward_delta_to_spot(delta_f, F, S)
        candidates.append({
            'strike': strike, 'log_moneyness': k, 'iv': sigma,
            'delta_forward': delta_f, 'delta_spot': delta_s,
            'delta_error': abs(delta_s - target_delta),
        })
    if not candidates:
        raise ValueError('strikes cannot be empty')
    return min(candidates, key=lambda item: (item['delta_error'], item['strike']))

## 测试：解析解与中心差分交叉验证

In [ ]:
S, K, sigma, tau, r, q = 6500.0, 6600.0, 0.22, 45 / 365, 0.015, 0.008
F = S * math.exp((r - q) * tau)
option_type = 'CALL'
h_F, h_S, h_vol, h_rate, h_tau = 0.1, 0.1, 1e-5, 1e-6, 1e-6

price = black76_price(F, K, sigma, tau, r, option_type)
put_price = black76_price(F, K, sigma, tau, r, 'PUT')
parity_error = price - put_price - math.exp(-r * tau) * (F - K)

price_F = lambda x: black76_price(x, K, sigma, tau, r, option_type)
delta_fd = (price_F(F + h_F) - price_F(F - h_F)) / (2 * h_F)
gamma_fd = (price_F(F + h_F) - 2 * price_F(F) + price_F(F - h_F)) / h_F**2

price_vol = lambda x: black76_price(F, K, x, tau, r, option_type)
vega_fd = (price_vol(sigma + h_vol) - price_vol(sigma - h_vol)) / (2 * h_vol)
volga_fd = (price_vol(sigma + h_vol) - 2 * price_vol(sigma) + price_vol(sigma - h_vol)) / h_vol**2

def vega_at_spot(spot):
    forward = spot * math.exp((r - q) * tau)
    return black76_vega(forward, K, sigma, tau, r)
vanna_fd = (vega_at_spot(S + h_S) - vega_at_spot(S - h_S)) / (2 * h_S)

def price_at_spot(spot):
    forward = spot * math.exp((r - q) * tau)
    return black76_price(forward, K, sigma, tau, r, option_type)
spot_delta_fd = (price_at_spot(S + h_S) - price_at_spot(S - h_S)) / (2 * h_S)
spot_gamma_fd = (price_at_spot(S + h_S) - 2 * price_at_spot(S) + price_at_spot(S - h_S)) / h_S**2

def price_at_repo(repo):
    forward = S * math.exp((r - repo) * tau)
    return black76_price(forward, K, sigma, tau, r, option_type)
repo_rho_fd = (price_at_repo(q + h_rate) - price_at_repo(q - h_rate)) / (2 * h_rate)

def price_at_rate(rate):
    forward = S * math.exp((rate - q) * tau)
    return black76_price(forward, K, sigma, tau, rate, option_type)
rho_fd = (price_at_rate(r + h_rate) - price_at_rate(r - h_rate)) / (2 * h_rate)

def price_at_tau(t):
    forward = S * math.exp((r - q) * t)
    return black76_price(forward, K, sigma, t, r, option_type)
theta_fd = (price_at_tau(tau + h_tau) - price_at_tau(tau - h_tau)) / (2 * h_tau)

delta_f = black76_delta(F, K, sigma, tau, r, option_type)
gamma_f = black76_gamma(F, K, sigma, tau, r)
tests = [
    ('Put-call parity', parity_error, 0.0, 1e-9),
    ('Forward Delta', delta_f, delta_fd, 1e-7),
    ('Forward Gamma', gamma_f, gamma_fd, 1e-7),
    ('Vega', black76_vega(F, K, sigma, tau, r), vega_fd, 1e-5),
    ('Volga', black76_volga(F, K, sigma, tau, r), volga_fd, 2e-2),
    ('Vanna', black76_vanna(F, K, sigma, tau, r, S), vanna_fd, 1e-5),
    ('Spot Delta conversion', forward_delta_to_spot(delta_f, F, S), spot_delta_fd, 1e-7),
    ('Spot Gamma conversion', forward_gamma_to_spot(gamma_f, F, S), spot_gamma_fd, 1e-7),
    ('Repo Rho', black76_repo_rho(F, K, sigma, tau, r, option_type), repo_rho_fd, 1e-4),
    ('Rho', black76_rho(F, K, sigma, tau, r, option_type), rho_fd, 1e-4),
    ('Theta dV/dtau', black76_theta(F, K, sigma, tau, r, q, option_type), theta_fd, 1e-4),
]
test_table = pd.DataFrame([
    {'测试': name, '解析结果': analytic, '差分/目标结果': numeric,
     '绝对误差': abs(analytic - numeric), '容差': tolerance,
     '通过': abs(analytic - numeric) <= tolerance}
    for name, analytic, numeric, tolerance in tests
])
assert test_table['通过'].all(), test_table.loc[~test_table['通过']]
display(test_table)

In [ ]:
# 代码、到期日、OTM、Delta搜索与曲面差分接口测试
option_parsed = parse_option_code('MO2606-C-6500.CFE')
future_parsed = parse_future_code('IM202407.CFE')
special_expiry = calculate_expiry_date('2606', {'2606': '2026-06-22'})
regular_expiry = calculate_expiry_date('2607')
assert option_parsed == {
    'expiry_code': '2606', 'expiry_year': 2026, 'expiry_month': 6,
    'option_type': 'CALL', 'strike': 6500.0,
}
assert future_parsed['expiry_year'] == 2024 and future_parsed['expiry_month'] == 7
assert special_expiry.isoformat() == '2026-06-22'
assert regular_expiry.isoformat() == '2026-07-17'
assert calculate_tau('2026-06-01', special_expiry) == 21 / 365
assert is_otm_option(6500, 6600, 'CALL')
assert is_otm_option(6500, 6400, 'PUT')

strike_result = find_strike_by_delta(
    strikes=range(5800, 7300, 100), target_delta=0.25,
    F=F, S=S, tau=tau, r=r, option_type='CALL',
    volatility_fn=lambda k: 0.22 + 0.02 * k + 0.10 * k * k,
)

# 人工多项式价值函数用于验证通用曲面差分接口
def mock_reprice(shocks):
    a = shocks.get('ATM', 0.0)
    s = shocks.get('Skew', 0.0)
    c = shocks.get('Curvature', 0.0)
    x = shocks.get('Spot', 0.0)
    return 100 + 2*a + 3*s + 4*c + 5*s*s + 6*s*c + 7*c*c + 8*x*s + 9*x*c

surface_test = calculate_vol_greeks(mock_reprice, spot_bump=0.1)
expected_surface = {
    'ATM': 2.0, 'Skew': 3.0, 'Curvature': 4.0,
    'VolgaSkew2': 10.0, 'VolgaSkewCurvature': 6.0,
    'VolgaCurvature2': 14.0, 'VannaSkew': 8.0, 'VannaCurvature': 9.0,
}
for key, expected in expected_surface.items():
    assert math.isclose(surface_test[key], expected, rel_tol=0, abs_tol=2e-6), (key, surface_test[key])

utility_test_table = pd.DataFrame([
    {'测试': '期权代码解析', '结果': option_parsed, '通过': True},
    {'测试': '期货代码解析', '结果': future_parsed, '通过': True},
    {'测试': '2606特殊到期日', '结果': special_expiry, '通过': True},
    {'测试': '2607第三个周五', '结果': regular_expiry, '通过': True},
    {'测试': 'Delta执行价搜索', '结果': strike_result, '通过': True},
    {'测试': '曲面中心差分接口', '结果': surface_test, '通过': True},
])
display(utility_test_table)

## 函数列表与数学定义

- `black76_price`：Black-76 Call/Put 价格。
- `black76_delta`、`black76_gamma`：对 Forward 的一阶、二阶导数。
- `black76_vega`、`black76_volga`：对局部/ATM 波动率的一级、二级敏感度。
- `black76_vanna`：$\partial^2V/(\partial S\partial\sigma)$。
- `black76_theta`：$\partial V/\partial\tau$；时间流逝使用负的 $\Delta\tau$。
- `black76_rho`、`black76_repo_rho`：Spot 口径的利率与 Repo 敏感度。
- `forward_delta_to_spot`、`forward_gamma_to_spot`：Forward 风险到 Spot 风险的链式转换。
- `calculate_vol_greeks`：固定实际合约，按曲面因子中心差分重估。
- `parse_option_code`、`parse_future_code`：支持 `yymm` 与 `yyyymm` 合约格式。
- `calculate_expiry_date`：特殊日期覆盖优先，否则取第三个周五。
- `calculate_tau`、`is_otm_option`、`find_strike_by_delta`：期限、OTM 与目标 Delta 工具。